Mount Google Drive and initialize project

In [2]:
from google.colab import drive
import os
import sys

# Mount Google Drive
drive.mount('/content/drive')

# Project configuration
PROJECT_DIR = "/content/drive/MyDrive/DumLLM"

# Set working directory
os.chdir(PROJECT_DIR)

print("Project mounted successfully!")
print("Working directory:", os.getcwd())
print("Python version:", sys.version)


Mounted at /content/drive
Project mounted successfully!
Working directory: /content/drive/MyDrive/DumLLM
Python version: 3.13.18 (main, Oct 26 2026, 15:28:15) [GCC 13.3.0]


#GitHub Auto Sync

In [ ]:

import os
import subprocess
import base64
from google.colab import userdata

# ==========================================
# DumLLM - GitHub Auto Sync
# ==========================================

PROJECT_DIR = "/content/drive/MyDrive/DumLLM"
REPO_URL = "https://github.com/ShubhamKumarSahu-svg/DumLLM.git"
BRANCH = "main"

os.chdir(PROJECT_DIR)


def git(command, check=True):
    result = subprocess.run(
        ["git"] + command,
        cwd=PROJECT_DIR,
        capture_output=True,
        text=True
    )

    if check and result.returncode != 0:
        raise RuntimeError(result.stderr)

    return result


# 1. Verify repository
if not os.path.isdir(".git"):
    raise RuntimeError("Git repository not found!")

# 2. Configure Git
git(["config", "user.name", "Shubham Kumar Sahu"])
git(["config", "user.email", "YOUR_GITHUB_EMAIL"])

# 3. Configure remote
git(["remote", "set-url", "origin", REPO_URL])

# 4. Protect large files and secrets
ignore_rules = [
    ".venv/",
    "venv/",
    "__pycache__/",
    "*.py[cod]",
    ".env",
    ".env.*",
    "!.env.example",
    ".ipynb_checkpoints/",
    "/gpt2/",
    "/model.pth",
    "checkpoints/",
    "*.pth",
    "*.ckpt",
    "*.safetensors"
]

existing = ""
if os.path.exists(".gitignore"):
    with open(".gitignore", "r") as f:
        existing = f.read()

with open(".gitignore", "a") as f:
    for rule in ignore_rules:
        if rule not in existing.splitlines():
            f.write("\n" + rule)

# 5. Check authentication BEFORE committing
token = userdata.get("GITHUB_TOKEN")

if not token:
    raise RuntimeError("GITHUB_TOKEN missing from Colab Secrets!")

auth = base64.b64encode(
    f"x-access-token:{token}".encode()
).decode()

# 6. Ask for commit message
message = input("Enter commit message: ").strip()

if not message:
    raise ValueError("Commit message cannot be empty!")

# 7. Fetch latest GitHub changes
git(["fetch", "origin"])

# 8. Ensure local branch is synchronized
status = git(["status", "-sb"])
print(status.stdout)

# Refuse to overwrite remote history
merge_base = git(
    ["merge-base", "HEAD", f"origin/{BRANCH}"],
    check=False
)

if merge_base.returncode != 0:
    raise RuntimeError(
        "Local and remote histories have diverged. "
        "Resolve synchronization before committing."
    )

# 9. Stage changes
git(["add", "-A"])

# 10. Check staged files for oversized files
files = git(["diff", "--cached", "--name-only"]).stdout.splitlines()

for file in files:
    result = subprocess.run(
        ["git", "cat-file", "-s", f":{file}"],
        cwd=PROJECT_DIR,
        capture_output=True,
        text=True
    )

    if result.returncode == 0:
        size = int(result.stdout.strip())
        if size > 95 * 1024 * 1024:
            raise RuntimeError(
                f"Large file detected: {file} ({size / 1024**2:.1f} MB). "
                "Remove it from Git tracking before committing."
            )

# 11. Commit if changes exist
changes = git(["diff", "--cached", "--quiet"], check=False)

if changes.returncode == 1:
    git(["commit", "-m", message])
    print("Commit created!")
elif changes.returncode == 0:
    print("No new changes to commit.")
else:
    raise RuntimeError("Could not inspect staged changes.")

# 12. Push all local commits
result = subprocess.run(
    [
        "git",
        "-c", f"http.extraheader=AUTHORIZATION: basic {auth}",
        "push", "-u", "origin", f"HEAD:{BRANCH}"
    ],
    cwd=PROJECT_DIR,
    capture_output=True,
    text=True
)

if result.returncode != 0:
    print(result.stderr)
    raise RuntimeError(
        "Push failed. Local commits are preserved. "
        "Resolve the reported Git error before retrying."
    )

print("\n" + "=" * 45)
print("DumLLM GitHub Sync Successful!")
print("=" * 45)
print("Commit:", message)
print("Branch:", BRANCH)
print("Repository:", REPO_URL)
print("\nLatest commit:")
print(git(["log", "-1", "--oneline"]).stdout)


Enter commit message: feat(tuning): prepare the datasets
